# Lesson 2: GDS Workflows in Python

**Duration:** ~15 minutes  
**Module:** GDS with Python  
**Dataset:** Movies (Actors, Movies, Genres, Users)

## What You'll Learn

By the end of this notebook, you'll be able to:

- Execute the standard five-step GDS workflow in Python
- Load data from CSV files programmatically
- Create graph projections and inspect them using the Graph object
- Choose the right execution mode for different situations
- Work with algorithm results as pandas DataFrames
- Clean up projections properly to manage memory

## The GDS Workflow

Every GDS analysis follows the same five steps:

1. **Load data** into Neo4j (if needed)
2. **Project** the graph into GDS memory
3. **Run algorithms** on the projection
4. **Work with results**
5. **Drop** the projection

This is the same workflow you used in Modules 1 and 2 with Cypher. The logic hasn't changed—only the interface.

## Setup: Connect to Neo4j

Your notebook is already connected to a dockerised Neo4j Database.

However, if you were using Desktop or Aura, you would need to connect using the correct environment variables below.

This is the same setup from Lesson 1.

In [4]:
import os
import pandas as pd
from IPython.display import display
from graphdatascience import GraphDataScience
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

uri = os.getenv('NEO4J_URI')
username = os.getenv('NEO4J_USERNAME')
password = os.getenv('NEO4J_PASSWORD')

# Create GDS client
gds = GraphDataScience(uri, auth=(username, password))

print(f"Connected to GDS version: {gds.version()}")

Connected to GDS version: 2.25.0


## Step 1: Loading Data

The `gds.run_cypher()` method executes any Cypher query and returns results as a pandas DataFrame. This makes it useful for data loading, ad-hoc queries, and anything else you'd normally do in the Browser.

We'll load a movies dataset with four node types and three relationship types:

| Nodes | Relationships |
|-------|---------------|
| `Movie`, `Actor`, `User`, `Genre` | `ACTED_IN`, `RATED`, `IN_GENRE` |

The CSV files are hosted on Google Drive. Let's configure the URLs first.

In [5]:
# CSV file URLs (Google Drive direct download format)
CSV_URLS = {
    'movies': 'https://drive.google.com/uc?export=download&id=1gqAoC1rFoiALglxXVhteJg-tQhiuxMgX',
    'actors': 'https://drive.google.com/uc?export=download&id=1XZDcSaCF5wSttZnl4kpfiouWTurp_OXR',
    'users': 'https://drive.google.com/uc?export=download&id=12CUzYiVmwQ0_No8sZtct6nb6D2uCl9nP',
    'genres': 'https://drive.google.com/uc?export=download&id=1s-eO-qohiSQpSzInP0NBgG0qfNn3wstK',
    'acted_in': 'https://drive.google.com/uc?export=download&id=1HR3tcJquCZWVthHdIZkOs31Mf2IIuwHP',
    'rated': 'https://drive.google.com/uc?export=download&id=1S7X006ktgUTDsWUhhKvcRPP6a26t8Ufl',
    'in_genre': 'https://drive.google.com/uc?export=download&id=1Zucm-Bq1mrYR3narTAZgSfm9dABRTXkP'
}

print("CSV URLs configured.")

CSV URLs configured.


Before loading new data, let's clear any existing data. This ensures the notebook runs consistently regardless of what's already in the database.

In [6]:
# Clear existing data for repeatability
gds.run_cypher("MATCH (n) DETACH DELETE n")
print("Database cleared.")

Database cleared.


Now let's load the Movie nodes. Notice how `gds.run_cypher()` returns a DataFrame, which we can use to confirm how many nodes were created.

In [7]:
# Load Movie nodes
result = gds.run_cypher(f"""
    LOAD CSV WITH HEADERS FROM '{CSV_URLS['movies']}' AS row
    MERGE (m:Movie {{tmdbId: row.tmdbId}})
    SET m.title = row.title,
        m.year = toInteger(row.year),
        m.plot = row.plot,
        m.runtime = toInteger(row.runtime),
        m.imdbRating = toFloat(row.imdbRating),
        m.imdbVotes = toInteger(row.imdbVotes),
        m.poster = row.poster,
        m.revenue = toInteger(row.revenue),
        m.budget = toInteger(row.budget),
        m.released = row.release
    RETURN count(m) AS movies_loaded
""")
print(f"Loaded {result['movies_loaded'][0]} movies")

Loaded 9125 movies


The pattern is the same for each node and relationship type. Let's load the rest in a single cell.

In [8]:
# Load Actor nodes
result = gds.run_cypher(f"""
    LOAD CSV WITH HEADERS FROM '{CSV_URLS['actors']}' AS row
    MERGE (a:Actor {{tmdbId: row.tmdbId}})
    SET a.name = row.name,
        a.born = toInteger(row.born),
        a.bornIn = row.bornIn,
        a.died = toInteger(row.died),
        a.bio = row.bio,
        a.poster = row.poster
    RETURN count(a) AS actors_loaded
""")
print(f"Loaded {result['actors_loaded'][0]} actors")

# Load User nodes
result = gds.run_cypher(f"""
    LOAD CSV WITH HEADERS FROM '{CSV_URLS['users']}' AS row
    MERGE (u:User {{userId: row.userId}})
    SET u.name = row.name
    RETURN count(u) AS users_loaded
""")
print(f"Loaded {result['users_loaded'][0]} users")

# Load Genre nodes
result = gds.run_cypher(f"""
    LOAD CSV WITH HEADERS FROM '{CSV_URLS['genres']}' AS row
    MERGE (g:Genre {{name: row.genre}})
    RETURN count(g) AS genres_loaded
""")
print(f"Loaded {result['genres_loaded'][0]} genres")

# Load ACTED_IN relationships
result = gds.run_cypher(f"""
    LOAD CSV WITH HEADERS FROM '{CSV_URLS['acted_in']}' AS row
    MATCH (a:Actor {{tmdbId: row.tmdbIdOut}})
    MATCH (m:Movie {{tmdbId: row.tmdbIdIn}})
    MERGE (a)-[:ACTED_IN]->(m)
    RETURN count(*) AS rels_created
""")
print(f"Created {result['rels_created'][0]} ACTED_IN relationships")

# Load RATED relationships
result = gds.run_cypher(f"""
    LOAD CSV WITH HEADERS FROM '{CSV_URLS['rated']}' AS row
    MATCH (u:User {{userId: row.userId}})
    MATCH (m:Movie {{tmdbId: row.tmdbId}})
    MERGE (u)-[r:RATED]->(m)
    SET r.rating = toFloat(row.rating),
        r.timestamp = toInteger(row.timestamp)
    RETURN count(r) AS rels_created
""")
print(f"Created {result['rels_created'][0]} RATED relationships")

# Load IN_GENRE relationships
result = gds.run_cypher(f"""
    LOAD CSV WITH HEADERS FROM '{CSV_URLS['in_genre']}' AS row
    MATCH (m:Movie {{tmdbId: row.tmdbId}})
    MATCH (g:Genre {{name: row.genre}})
    MERGE (m)-[:IN_GENRE]->(g)
    RETURN count(*) AS rels_created
""")
print(f"Created {result['rels_created'][0]} IN_GENRE relationships")

Loaded 15443 actors
Loaded 671 users
Loaded 20 genres
Created 35910 ACTED_IN relationships
Created 100004 RATED relationships
Created 20340 IN_GENRE relationships


Let's take a quick look at the data to confirm everything loaded correctly.

In [ ]:
# Sample the data
df = gds.run_cypher("""
    MATCH (a:Actor)-[:ACTED_IN]->(m:Movie)
    RETURN a.name AS actor, m.title AS movie, m.year AS year
    LIMIT 5
""")
print("Sample data:")
display(df)

**What just happened?**

- `gds.run_cypher()` executed Cypher queries from Python
- Each query returned results as a pandas DataFrame
- We accessed values using DataFrame syntax: `result['column_name'][0]`

This method works for any Cypher query—not just data loading. You can use it for ad-hoc exploration, schema inspection, or anything else you'd normally do in the Browser.

## Step 2: Creating Projections

The `gds.graph.project()` method returns two values:

1. A **Graph object** (`G`) for interacting with the projection
2. **Metadata** about what was projected

Let's project actors and movies connected by `ACTED_IN` relationships. We'll include some node properties as well, using `defaultValue` to handle any null values.

In [ ]:
# Create a graph projection with properties
G, result = gds.graph.project(
    "movies-graph",
    {
        "Actor": {
            "properties": {
                "born": {"defaultValue": 1900}
            }
        },
        "Movie": {
            "properties": {
                "year": {"defaultValue": 1900},
                "imdbRating": {"defaultValue": 0.0},
                "runtime": {"defaultValue": 90}
            }
        }
    },
    "ACTED_IN"
)

print(f"Nodes: {G.node_count()}")
print(f"Relationships: {G.relationship_count()}")

### Inspecting the Projection with the Graph Object

The Graph object provides methods to inspect your projection without querying the catalog directly. This is useful for verifying that your projection contains what you expect before running algorithms.

In [ ]:
# Explore the Graph object
print(f"Graph name: {G.name()}")
print(f"Node count: {G.node_count()}")
print(f"Relationship count: {G.relationship_count()}")
print(f"Node labels: {G.node_labels()}")
print(f"Relationship types: {G.relationship_types()}")
print(f"Memory usage: {G.memory_usage()}")
print(f"Exists in catalog: {G.exists()}")

In [ ]:
# Check which properties are available on each node label
print(f"Movie node properties: {G.node_properties('Movie')}")
print(f"Actor node properties: {G.node_properties('Actor')}")

## Step 3: Running Algorithms

Algorithm calls follow a consistent pattern:

```python
gds.<algorithm>.<mode>(G, **config)
```

The **mode** determines what happens with the results. Let's look at each one.

### Stream Mode

Stream mode returns results as a pandas DataFrame. This is the most common choice when you want to analyze or visualize results in Python.

In [ ]:
# Stream degree centrality results
df = gds.degree.stream(G)

print("Degree centrality results (top 10):")
display(df.nlargest(10, "score"))

### Mutate Mode

Mutate mode stores results in the projection itself. The results aren't written to Neo4j—they exist only in memory. This is useful when you want to chain algorithms together or use the results in subsequent GDS operations.

In [ ]:
# Mutate: store degree centrality in the projection
result = gds.degree.mutate(
    G,
    mutateProperty="rank"
)

print(f"Nodes processed: {result['nodePropertiesWritten']}")

# The property now exists on nodes in the projection
print(f"Actor properties now: {G.node_properties('Actor')}")

### Write Mode

Write mode persists results to Neo4j. Use this when you need the results available for future queries or want to visualize them in the Browser.

In [ ]:
# Write: store degree centrality in the database
result = gds.degree.write(
    G,
    writeProperty="degree"
)

print(f"Wrote degree centrality to {result['nodePropertiesWritten']} nodes")

Let's verify the write worked by querying the database directly.

In [ ]:
# Verify by querying the database
df = gds.run_cypher("""
    MATCH (a:Actor)
    WHERE a.degree IS NOT NULL
    RETURN a.name AS actor, a.degree AS degree
    ORDER BY degree DESC
    LIMIT 5
""")
print("Top actors by degree (from database):")
display(df)

### Stats Mode

Stats mode returns summary statistics without storing results anywhere. This is useful for quick checks or when you only need aggregate information.

In [ ]:
# Stats: get statistics only
result = gds.degree.stats(G)

print(f"Centrality distribution: {result['centralityDistribution']}")
print(f"Compute time (ms): {result['computeMillis']}")

## Step 4: Working with Results

Since stream mode returns DataFrames, you can use the full pandas toolkit for analysis. Let's look at a few common patterns.

In [ ]:
# Get degree centrality scores as a DataFrame
scores = gds.degree.stream(G)

# Find the top 10 nodes
top_nodes = scores.nlargest(10, "score")
print("Top 10 nodes by degree:")
display(top_nodes)

The DataFrame contains `nodeId` values, which are internal Neo4j identifiers. To get meaningful node information (like names), we need to look them up in the database.

In [ ]:
# Get node names from nodeIds
top_with_names = gds.run_cypher("""
    MATCH (n)
    WHERE id(n) IN $nodeIds
    RETURN id(n) AS nodeId, 
           CASE WHEN n:Actor THEN n.name ELSE n.title END AS name,
           labels(n)[0] AS label
""", params={"nodeIds": top_nodes["nodeId"].tolist()})

# Merge the names with the scores
result = top_nodes.merge(top_with_names, on="nodeId")
print("Top nodes with names:")
display(result[["name", "label", "score"]])

## Step 5: Cleanup

Projections consume memory. When you're finished with a projection, drop it to free those resources. This is especially important in notebooks, where you might create multiple projections during exploration.

In [ ]:
# Check what projections currently exist
print("Current projections:")
display(gds.graph.list())

In [ ]:
# Drop the projection
G.drop()
print("Projection dropped.")

# Verify it's gone
print(f"Remaining projections: {len(gds.graph.list())}")

### The Context Manager Pattern

Python's `with` statement provides automatic cleanup. When the block ends, the projection is dropped—even if an error occurs. This pattern is especially useful for exploratory work.

In [ ]:
# Context manager automatically drops the projection when the block ends
with gds.graph.project("temp-graph", ["User", "Movie"], "RATED")[0] as G:
    result = gds.degree.stream(G)
    print(f"Ran degree centrality on {G.node_count()} nodes")
    print("Top 5 by degree:")
    display(result.nlargest(5, "score"))

# G is automatically dropped here
print(f"\nGraph still exists? {gds.graph.exists('temp-graph')['exists']}")

## Summary

You've now worked through the complete GDS workflow in Python:

| Step | Method | What It Does |
|------|--------|-------------|
| Load | `gds.run_cypher()` | Execute Cypher, get DataFrames |
| Project | `gds.graph.project()` | Create in-memory graph |
| Inspect | `G.node_count()`, etc. | Examine projection |
| Run | `gds.<algo>.<mode>(G)` | Execute algorithms |
| Cleanup | `G.drop()` | Free memory |

**Key takeaways:**

- `gds.graph.project()` returns a Graph object for easy inspection
- Four execution modes let you choose where results go: `.stream()`, `.mutate()`, `.write()`, `.stats()`
- Always drop projections when finished—or use context managers for automatic cleanup
- Include `defaultValue` when projecting properties to handle nulls

**Next lesson:** Understanding projection syntax options—native vs. Cypher projection in Python.

In [ ]:
# Close the connection when done
gds.close()